### Create Bronze storage

In [0]:
from pyspark.sql import functions as F

# Store raw ingested transactions in a dedicated schema.
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.sentinel_bronze
""")

# Keep ingestion state separate from incoming transaction files.
spark.sql("""
    CREATE VOLUME IF NOT EXISTS
    workspace.sentinel_bronze.ingestion_state
""")

landing_path = (
    "/Volumes/workspace/sentinel_sources/replay_landing"
)

state_path = (
    "/Volumes/workspace/sentinel_bronze/"
    "ingestion_state/transactions_v1"
)

bronze_table = "workspace.sentinel_bronze.transactions"

### Ingest released files

In [0]:
# Auto Loader discovers Parquet files in landing.
# The partition folder supplies replay_batch.
# schemaLocation preserves the discovered source schema.
incoming = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{state_path}/schema")
    .option("cloudFiles.partitionColumns", "replay_batch")
    .load(landing_path)
)

# Preserve source fields and add ingestion traceability.
# This path identifies the released Parquet file;
# source_file still identifies the original CSV.
bronze = incoming.select(
    "*",
    F.col("_metadata.file_path").alias("_ingested_file_path"),
    F.current_timestamp().alias("_ingestion_timestamp")
)

# Append to a managed Delta table.
# The checkpoint records progress across executions.
# AvailableNow processes available files and then stops.
query = (
    bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{state_path}/checkpoint")
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()

print("Bronze ingestion completed.")

### Verify Bronze ingestion against the manifest

**Purpose**: Confirm that `workspace.sentinel_bronze.transactions` matches the expected row count in `/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest` for every released batch in `/Volumes/workspace/sentinel_sources/replay_landing`.

**Small example**:
- When only `2019-01-01` is released, the manifest expects `2,414` rows.
- When `2019-01-01` through `2019-01-28` are released (`28` batches), the manifest expects `47,875` total rows across those dates.

In [0]:
manifest_path = (
    "/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest"
)

# Read the persisted Bronze table and the replay manifest.
saved_bronze = spark.table(bronze_table)
manifest = spark.read.parquet(manifest_path)

# Discover which daily batches have been released into landing.
released_batches = (
    spark.read.parquet(landing_path)
    .select("replay_batch")
    .distinct()
)

# Join the manifest with released batches to get expected counts.
expected_by_batch = manifest.join(released_batches, on="replay_batch", how="inner")
actual_by_batch = (
    saved_bronze
    .groupBy("replay_batch")
    .agg(F.count("*").alias("actual_transaction_count"))
)

verification = (
    expected_by_batch
    .join(actual_by_batch, on="replay_batch", how="full_outer")
    .withColumn(
        "matches_manifest",
        F.col("actual_transaction_count") == F.col("expected_transaction_count"),
    )
    .orderBy("replay_batch")
)

mismatches = verification.filter(
    F.col("matches_manifest").isNull() | (~F.col("matches_manifest"))
)
assert mismatches.count() == 0, (
    "Bronze daily counts do not match the manifest for released batches"
)

expected_total = (
    expected_by_batch
    .agg(F.sum("expected_transaction_count").alias("total"))
    .collect()[0]["total"]
)
actual_total = saved_bronze.count()
distinct_ids = saved_bronze.select("trans_num").distinct().count()

assert actual_total == expected_total, (
    f"Expected {expected_total:,} Bronze transactions, found {actual_total:,}"
)
assert distinct_ids == expected_total, (
    f"Expected {expected_total:,} distinct transaction IDs, found {distinct_ids:,}"
)
assert "is_fraud" not in saved_bronze.columns, (
    "Fraud labels must not appear in Bronze"
)
assert saved_bronze.filter(F.col("replay_batch") >= F.lit("2019-01-29")).count() == 0, (
    "Evaluation batches (2019-01-29 onward) must not appear in Bronze"
)

display(verification)

print(
    f"Verified {expected_by_batch.count()} released batch(es): "
    f"{actual_total:,} Bronze transactions with unique IDs matching the manifest."
)

## Bronze ingestion verification

- Released batches in `replay_landing` are ingested incrementally using the Auto Loader checkpoint.
- Per-day and total Bronze row counts match `/Volumes/workspace/sentinel_sources/replay_staging/replay_v1/manifest` (`2,414` on January 1; `47,875` across January 1–28).
- Distinct transaction IDs match the expected manifest total.
- Fraud labels are excluded from Bronze.
- Ingestion file path and timestamp are recorded on every row.
- Rerunning ingestion with the same checkpoint adds zero duplicate rows.
- Evaluation batches (`2019-01-29` onward) remain unreleased.